In [ ]:
import tensorflow as tf 
import keras 
from segmentation_preprocessing import train_ds, valid_ds,combined_loss, dice_metric

tf.random.set_seed(0)

he_init=keras.initializers.HeNormal()

elu_act=keras.activations.elu

l2_reg=keras.regularizers.l2(1e-4)

inputs=keras.layers.Input(shape=(128,128,3))

# Encoder which downscales the image
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(inputs)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.MaxPool2D((2,2))(x)
block_1_output=x

x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.MaxPool2D((2,2))(x)
block_2_output=x

x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.MaxPool2D((2,2))(x)
block_3_output=x

x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=256, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.MaxPool2D((2,2))(x)
block_4_output=x

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(block_4_output)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

x=keras.layers.Conv2D(512, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Dropout(0.1)(x)

bottleneck=x

# Decoder which upscales the images

# 2x2 kernel upscaled from H->2H
x=keras.layers.Conv2DTranspose(filters=128,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(bottleneck)
x=keras.layers.Concatenate()([x,block_3_output]) # Up from 4->3

x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=128, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)

x=keras.layers.Conv2DTranspose(filters=64,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_2_output])

x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=64, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)

x=keras.layers.Conv2DTranspose(filters=32,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)
x=keras.layers.Concatenate()([x,block_1_output])

x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=32, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)

x=keras.layers.Conv2DTranspose(filters=16,kernel_size=(2,2),strides=2, padding="same", kernel_initializer=he_init, kernel_regularizer=l2_reg)(x)

x=keras.layers.Conv2D(filters=16, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)
x=keras.layers.Conv2D(filters=16, kernel_size=(3,3), kernel_initializer=he_init, kernel_regularizer=l2_reg, padding="same")(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Activation(elu_act)(x)

outputs=keras.layers.Conv2D(1, kernel_size=(1,1), activation='sigmoid')(x)

model=keras.Model(inputs, outputs)

adam=keras.optimizers.Adam(1e-4, clipnorm=1.0)

model.compile(
    optimizer=adam,
    loss=combined_loss,
    metrics=[dice_metric]
)

earlyStop_cb=keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    verbose=1,
    restore_best_weights=True,
    mode='min')

history=model.fit(
    train_ds,
    batch_size=32,
    epochs=20,
    callbacks=[earlyStop_cb],
    validation_data=valid_ds
)

model.save('Saved Models/Seg_ELU_L2_CombinedLoss_DiceMetric.keras')

2026-02-17 00:06:16.167190: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-02-17 00:06:16.167215: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-02-17 00:06:16.167218: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-02-17 00:06:16.167234: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-02-17 00:06:16.167244: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
2026-02-17 00:06:17.710342: W tensorflow/core/kernels/data/cache_dataset_ops.cc:302] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncat

tf.Tensor(0.0, shape=(), dtype=float32) tf.Tensor(1.0, shape=(), dtype=float32)


2026-02-17 00:06:17.997558: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


Epoch 1/20


2026-02-17 00:06:20.540397: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


57/57 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - dice_metric: 0.4190 - loss: 1.9189

2026-02-17 00:08:50.628757: W tensorflow/core/kernels/data/cache_dataset_ops.cc:302] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


57/57 ━━━━━━━━━━━━━━━━━━━━ 153s 2s/step - dice_metric: 0.5098 - loss: 1.7702 - val_dice_metric: 0.4058 - val_loss: 2.1852
Epoch 2/20


2026-02-17 00:08:51.572657: W tensorflow/core/kernels/data/cache_dataset_ops.cc:302] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


57/57 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - dice_metric: 0.6693 - loss: 1.5027 - val_dice_metric: 0.6860 - val_loss: 1.3989
Epoch 3/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - dice_metric: 0.7364 - loss: 1.3727 - val_dice_metric: 0.7102 - val_loss: 1.3037
Epoch 4/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 82s 1s/step - dice_metric: 0.7636 - loss: 1.3063 - val_dice_metric: 0.7765 - val_loss: 1.1808
Epoch 5/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 69s 1s/step - dice_metric: 0.7850 - loss: 1.2471 - val_dice_metric: 0.8013 - val_loss: 1.1329
Epoch 6/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 77s 1s/step - dice_metric: 0.7987 - loss: 1.2121 - val_dice_metric: 0.8145 - val_loss: 1.0913
Epoch 7/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 79s 1s/step - dice_metric: 0.8096 - loss: 1.1740 - val_dice_metric: 0.8254 - val_loss: 1.0706
Epoch 8/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 72s 1s/step - dice_metric: 0.8192 - loss: 1.1362 - val_dice_metric: 0.8217 - val_loss: 1.0786
Epoch 9/20
57/57 ━━━━━━━━━━━━━━━━━━━━ 74s 1s/step - dice_metric: 0.8254 - loss: 1.1097 